# Imports

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import mne
import xml.etree.ElementTree as ET

# Paths

In [2]:
SIGNALS_PATH = r"C:\Users\waheed\Desktop\Sleep_Stages_Classification\Dataset\Signals"
ANNOT_PATH = r"C:\Users\waheed\Desktop\Sleep_Stages_Classification\Dataset\Annotations"

# Check dataset files

In [3]:
print("Annotations files:")
print(os.listdir(ANNOT_PATH))

print("\nSignals files:")
print(os.listdir(SIGNALS_PATH))

Annotations files:
['mesa-sleep-0001-nsrr.xml', 'mesa-sleep-0002-nsrr.xml', 'mesa-sleep-0006-nsrr.xml', 'mesa-sleep-0010-nsrr.xml', 'mesa-sleep-0012-nsrr.xml', 'mesa-sleep-0014-nsrr.xml', 'mesa-sleep-0016-nsrr.xml', 'mesa-sleep-0021-nsrr.xml', 'mesa-sleep-0027-nsrr.xml', 'mesa-sleep-0028-nsrr.xml']

Signals files:
['mesa-sleep-0001.edf', 'mesa-sleep-0002.edf', 'mesa-sleep-0006.edf', 'mesa-sleep-0010.edf', 'mesa-sleep-0012.edf', 'mesa-sleep-0014.edf', 'mesa-sleep-0016.edf', 'mesa-sleep-0021.edf', 'mesa-sleep-0027.edf', 'mesa-sleep-0028.edf']


# Load one subject

In [4]:
subject_id = "0001"

edf_file = os.path.join(SIGNALS_PATH, f"mesa-sleep-{subject_id}.edf")
xml_file = os.path.join(ANNOT_PATH, f"mesa-sleep-{subject_id}-nsrr.xml")

raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)

print(raw)
print("\nChannels:", raw.ch_names)

<RawEDF | mesa-sleep-0001.edf, 27 x 11058944 (43199.0 s), ~2.22 GiB, data loaded>

Channels: ['EKG', 'EOG-L', 'EOG-R', 'EMG', 'EEG1', 'EEG2', 'EEG3', 'Pres', 'Flow', 'Snore', 'Thor', 'Abdo', 'Leg', 'Therm', 'Pos', 'EKG_Off', 'EOG-L_Off', 'EOG-R_Off', 'EMG_Off', 'EEG1_Off', 'EEG2_Off', 'EEG3_Off', 'Pleth', 'OxStatus', 'SpO2', 'HR', 'DHR']


# Visualize signals

In [ ]:
raw.plot(duration=5, n_channels=3)

# EOG-R signal

In [ ]:
signal = raw.get_data(picks="EOG-R")[0]

plt.figure(figsize=(15,4))
plt.plot(signal[:1000])
plt.title("Raw EOG-R Signal (Sample)")
plt.xlabel("Samples")
plt.show()

# EOG shape

In [ ]:
eog = raw.get_data(picks='EOG-R')
print("EOG shape:", eog.shape)

# Explore Sleep Stages from XML

In [ ]:
tree = ET.parse(xml_file)
root = tree.getroot()

events = []

for e in root.iter('ScoredEvent'):
    c = e.find('EventConcept')
    if c is not None:
        events.append(c.text.strip())

print("Unique events:")
print(set(events))

# Convert annotations → labels

In [ ]:
SFREQ = 256
EPOCH_SEC = 30

In [ ]:
def parse_annotations(xml_path):
    tree = ET.parse(xml_path)
    stages = []
    
    for event in tree.getroot().iter('ScoredEvent'):
        concept = event.find('EventConcept')
        start = event.find('Start')
        duration = event.find('Duration')
        
        if concept is None or start is None or duration is None:
            continue
        
        text = concept.text.strip().split("|")[0]
        
        if text in LABEL_MAP:
            stages.append({
                "label": LABEL_MAP[text],
                "start": float(start.text),
                "duration": float(duration.text)
            })
    
    return stages

In [ ]:
def create_epoch_labels(stages, num_epochs):
    labels = np.full(num_epochs, -1, dtype=np.int64)
    
    for s in stages:
        start_ep = int(s["start"] // EPOCH_SEC)
        end_ep = int((s["start"] + s["duration"]) // EPOCH_SEC)
        
        for i in range(start_ep, min(end_ep, num_epochs)):
            labels[i] = s["label"]
    
    return labels

In [ ]:
stages = parse_annotations(xml_file)

num_epochs = len(signal) // (SFREQ * EPOCH_SEC)
labels = create_epoch_labels(stages, num_epochs)

print("Label distribution:")
print(Counter(labels))

# Plot class distribution

In [ ]:
counts = Counter(labels)

plt.bar(counts.keys(), counts.values())
plt.xticks([0,1,2,3,4], ["Wake","N1","N2","N3","REM"])
plt.title("Sleep Stage Distribution")
plt.show()